# Testing of random forest model transfer
This is a trial to test whether a model can be transferred from one classification purpose to other context.

In this step, the `MULTIPROBABILITY` output layers of the whole region (Sumatra in this case) have been generated and exported to local drive. This notebook's goal is to generate an LULC map from the pre-generated output layers in a smaller AOI

In [ ]:

import ee

ee.Authenticate()
ee.Initialize()

In [ ]:
import geemap

# Choose AOI

aoi = geemap.shp_to_ee('../data/AOI_Dempo.shp')

# Load default classification scheme

In [ ]:
# Choose default classification scheme

from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df)

# Load training data and identify the classes that exist in the AOI from default scheme

In [ ]:
# Load default training data from GEE asset and filter by AOI
# This is mainly to identify the existing classes inside the AOI and retrieve the class list information

from luma_ge.sample_data import SyncTrainData
import pandas as pd

TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=classification_df,
            aoi_geometry=aoi,
            training_ee_path=TrainEePath
        )

        # Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)
        
        # Check sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )

TrainDataFinal = TrainDataDict.get('training_data')

# Cross-check the class IDs in the filtered training data against the classification scheme
if TrainDataFinal is not None and hasattr(TrainDataFinal, 'columns') and 'kelas' in TrainDataFinal.columns:
    train_class_ids = pd.Series(TrainDataFinal['kelas'].dropna().astype(int).unique()).sort_values().tolist()
    scheme_df = classification_df[['ID', 'Land Cover Class']].copy()
    scheme_df['ID'] = scheme_df['ID'].astype(int)

    class_summary = []
    unmatched_ids = []

    for class_id in train_class_ids:
        scheme_match = scheme_df[scheme_df['ID'] == class_id]
        if not scheme_match.empty:
            class_name = scheme_match.iloc[0]['Land Cover Class']
            sample_count = int((TrainDataFinal['kelas'].astype(int) == class_id).sum())
            class_summary.append({
                'class_id': class_id,
                'class_name': class_name,
                'sample_count': sample_count
            })
        else:
            unmatched_ids.append(class_id)

    class_summary_df = pd.DataFrame(class_summary)
    display(class_summary_df)

    if unmatched_ids:
        print('Class IDs in training data but not present in classification_df:', unmatched_ids)
    else:
        print('All training class IDs are present in classification_df.')
else:
    print('No filtered training data available for class summary.')


# Load the pre-generated multiprobability image stack
Ideally the image stack should have been merged as the result of the exported job as GEE export split into smaller areas

In [ ]:
multiprobability_stack_path = '../data/temp/multiprobability_stack_example.tif'

import rasterio

with rasterio.open(multiprobability_stack_path) as src:
    descriptions = src.descriptions
    if descriptions and any(desc for desc in descriptions):
        band_names = [desc if desc else f'band_{i + 1}' for i, desc in enumerate(descriptions)]
    else:
        band_names = [f'band_{i + 1}' for i in range(src.count)]
    raster_meta = src.meta.copy()
    raster_data = src.read()

print('Available bands:')
for name in band_names:
    print(f' - {name}')

# Select only the bands whose names contain the class IDs from class_summary_df
if 'class_summary_df' in globals() and class_summary_df is not None and not class_summary_df.empty:
    class_ids = [str(int(cid)) for cid in class_summary_df['class_id'].astype(int).tolist()]
    selected_band_names = []
    selected_class_ids = []
    for name in band_names:
        for class_id in class_ids:
            if class_id in str(name):
                selected_band_names.append(name)
                selected_class_ids.append(int(class_id))
                break

    if not selected_band_names:
        selected_band_names = band_names[:min(len(band_names), len(class_ids))]
        selected_class_ids = [int(cid) for cid in class_ids[:len(selected_band_names)]]

    print('Selected bands for AOI classes:')
    for name in selected_band_names:
        print(f' - {name}')

    selected_band_indices = [i + 1 for i, name in enumerate(band_names) if name in selected_band_names]

    if not selected_band_indices:
        selected_band_indices = list(range(1, min(len(band_names), len(class_ids)) + 1))

    selected_stack = raster_data[[idx - 1 for idx in selected_band_indices]]
    raster_meta['count'] = len(selected_band_indices)
    print('Raster selection completed.')
else:
    print('class_summary_df is not available yet; run the training-data summary cell first.')


# Apply argmax function for the class probability layers

In [ ]:
import numpy as np

# Apply argmax to classify pixels based on the highest class probability
if selected_stack.ndim != 3:
    raise ValueError('selected_stack must be a 3D array with shape (bands, rows, cols).')

if len(selected_class_ids) != selected_stack.shape[0]:
    print('Warning: selected_class_ids length does not match selected_stack bands.')
    selected_class_ids = [i + 1 for i in range(selected_stack.shape[0])]

argmax_indices = np.argmax(selected_stack, axis=0)
classified_map = np.array([int(selected_class_ids[idx]) for idx in argmax_indices.flat]).reshape(argmax_indices.shape)

print('Argmax classification completed.')
print('classified_map shape:', classified_map.shape)

# Prepare classified raster metadata for output
classified_meta = raster_meta.copy()
classified_meta.update({
    'count': 1,
    'dtype': 'int16'
})

# Result visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
import tempfile
import os

# Visualize the classified map and class distribution
unique, counts = np.unique(classified_map, return_counts=True)
distribution_df = pd.DataFrame({'class_id': unique, 'pixel_count': counts})
distribution_df['percentage'] = 100 * distribution_df['pixel_count'] / distribution_df['pixel_count'].sum()

distribution_df = distribution_df.sort_values('class_id').reset_index(drop=True)

print('Bands included in final classification stack:')
for name in selected_band_names:
    print(f' - {name}')

print('\nFinal class distribution:')
print(distribution_df)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Classified map
im = ax[0].imshow(classified_map, cmap='tab20')
ax[0].set_title('Classified Map')
ax[0].axis('off')
fig.colorbar(im, ax=ax[0], label='Class ID')

# Distribution
colors = plt.cm.tab20(np.linspace(0, 1, len(distribution_df)))
ax[1].bar(distribution_df['class_id'].astype(str), distribution_df['pixel_count'], color=colors)
ax[1].set_title('Class Distribution')
ax[1].set_xlabel('Class ID')
ax[1].set_ylabel('Pixel Count')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('..data/temp/classified_map.png', dpi=150, bbox_inches='tight')
plt.show()  # Safe now